# 🩺 HealthSYNQ-D v8 — FINAL Production Engine
**Smartwatch-Based Adaptive Diet & Glucose Management System**

| Feature | Implementation |
|---|---|
| **Anti-repetition** | Hard-exclude week_used; Top-5 uniform random; cooldown fallback |
| **Category rotation** | Family-count penalty `−0.12/use` + recency penalty `−0.30` in last 4 |
| **Dal budget** | 40% of meal target → ALL 18 dals always eligible (was 20% = only 1 dal fit) |
| **Calorie** | Roti guaranteed 30%; `kcal_lo` at 88%; dynamic MAX for low-cal breakfast items |
| **Scoring** | `−0.4×carb − 0.3×GI + 0.2×protein + 0.1×fibre − 0.1×sugar` |
| **Glucose** | 5 tiers: Normal/Pre-diabetic/Diabetic/High/Very-high |
| **Type safety** | `week_used: dict[str, list[tuple]]` — single write point `_wu_add()` |

## ── CELL 1 ── Imports

In [1]:
import pandas as pd
import numpy as np
import random
import warnings
from collections import Counter
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 55)
pd.set_option('display.float_format', '{:.1f}'.format)
print('✅ Imports done')

✅ Imports done


## ── CELL 2 ── Load Dataset

In [2]:
DATASET_PATH = 'Anuvaad_INDB_2024.xlsx'  # ← update if needed

PER100G = ['food_code','food_name','servings_unit','energy_kcal','carb_g','protein_g','fat_g','freesugar_g','fibre_g']
PER_SRV = ['unit_serving_energy_kcal','unit_serving_carb_g','unit_serving_protein_g',
           'unit_serving_fat_g','unit_serving_freesugar_g','unit_serving_fibre_g']

raw = pd.read_excel(DATASET_PATH)
df  = raw[PER100G + PER_SRV].copy()
df['food_name']     = df['food_name'].str.strip()
df['servings_unit'] = df['servings_unit'].fillna('serving').astype(str)
df = df.dropna(subset=PER_SRV).reset_index(drop=True)
df = df.rename(columns={
    'unit_serving_energy_kcal': 'srv_kcal', 'unit_serving_carb_g':      'srv_carb',
    'unit_serving_protein_g':   'srv_protein','unit_serving_fat_g':       'srv_fat',
    'unit_serving_freesugar_g': 'srv_sugar', 'unit_serving_fibre_g':     'srv_fibre',
})
df = df[df['srv_kcal'] <= 900].reset_index(drop=True)
print(f'Loaded {len(df)} food items')

Loaded 813 food items


## ── CELL 3 ── Hard Remove

In [4]:
HARD_REMOVE = [
   r'\bcooler\b',r'lem-o',r'\biced tea\b',r'\bgin\b',r'\bvodka\b',
    r'whiskey',r'\brum\b',r'\bbeer\b',r'\bwine\b',r'\bpunch\b',r'\bnog\b',r'smoothie',
    r'\bsoup\b',r'\bstock\b',r'\bbroth\b',r'\bconsomm[eé]\b',
    r'\bbaby\b',r'\binfant\b',r'shishu',r'amylase',r'\bpowder\b',r'\bformula\b',r'\bsupplement\b',
    r'\bburfi\b',r'\bbarfi\b',r'\bladoo\b',r'\bpeda\b',r'\bmodak\b',r'\bchikki\b',
    r'peanut brittle',r'\bice cream\b',r'\bcake\b',r'\bpastry\b',r'\bpudding\b',
    r'\bbiscuit\b',r'\bcookie\b',r'\bpickle\b',r'\baachar\b',r'\bketchup\b',r'pickled mustard',
    r'\bjam\b',r'\bjelly\b',r'\bicing\b',r'\bgravy for\b',r'\bchips\b',r'\bpremix\b',
    r'\bspaghetti\b',r'\blasagne\b',r'\bpasta\b',r'\bnoodles\b',r'\bchowmein\b',r'\bpancake\b',
    r'\byakhni\b',r'\bwater\b',r'egg halwa',
    r'\bsouffle\b',r'\bpie\b',r'\bdrop\b',r'\bmathri\b',r'\bpuff\b',r'\bchop\b',
    r'buttermilk biscuit',
]
mask = df['food_name'].str.lower().str.contains('|'.join(HARD_REMOVE), regex=True)
df = df[~mask].reset_index(drop=True)
print(f'After cleaning: {len(df)} items')

After cleaning: 568 items


## ── CELL 4 ── Diet, Role, Grain & Sub-Category

In [3]:
NONVEG_KW = ['chicken','mutton','fish','prawn','keema','lamb','beef','pork',
             'crab','lobster','tuna','salmon','sardine','mackerel','shrimp',
             'meat ball','meatball','minced meat','scotch egg',
             'boti']         
EGG_KW    = ['boiled egg','fried egg','poached egg','scrambled egg','omelette',
             'omlet','egg curry','egg bhujia','anda bhujia','deviled egg',
             'baked egg','egg cutlet','egg sandwich','egg and tomato','egg pakora']
VEG_KOFTA = ['pea kofta','spinach kofta','paneer kofta','lotus stem kofta',
             'raw banana kofta','cauliflower kofta','cabbage kofta','lauki kofta',
             'ghiya kofta','vegetarian egg kofta','vegetarian nargisi kofta',
             'potato kofta','yam kofta','jackfruit kofta','spinach paneer kofta']

def classify_diet(name):
    n = name.lower()
    if any(k in n for k in VEG_KOFTA): return 'veg'
    if any(k in n for k in NONVEG_KW): return 'nonveg'
    if any(k in n for k in EGG_KW):    return 'egg'
    return 'veg'

def diet_filter(frame, diet):
    if diet == 'veg':
        return frame[frame['diet_type'] == 'veg']
    if diet == 'egg':
        return frame[frame['diet_type'].isin(['veg','egg'])]
    if diet == 'nonveg':
        return frame[frame['diet_type'].isin(['veg','egg','nonveg'])]
    return frame

ROLE_KW = {
    'roti': ['chapati','roti','parantha','paratha','naan','phulka','kulcha','thepla',
             'makki ki roti','bajra roti','jowar roti','akki roti','tandoori roti',
             'missi roti','laccha','paushtik roti','soya roti','puranpoli'
             ],
    'dal':  ['washed moong dal','washed urad dal','mixed dal','whole moong','whole masoor',
             'whole moth','whole urad','moti mahal dal','rajmah curry','kidney bean curry',
             'sambar','lobia curry','soyabean curry','black channa curry','chickpeas curry',
             'channa dal with','split bengal gram with','rajma','chole','chhole','arhar',
             'toor dal','moong ki dal','masoor ki dal','paneer curry','paneer masala','shahi paneer',
             'methi malai paneer','urad ki dal','moth ki dal',
             'dal makhani','matar paneer',
             'paneer butter masala','palak paneer','mushroom matar','matar mushroom','dal tadka','dal fry','panchmel dal','kadhai paneer','paneer lababdar','paneer in butter','veg paneer stew',
             'paneer stuffed cheela','paneer shaslik'],
    'sabzi':['aloo gobhi','aloo methi','aloo matar','aloo baingan','aloo palak',
             'shimla mirch aloo','sookhe aloo','dum aloo','bhindi','karela','lauki',
             'tinda','parval','baingan ka bhartha','brinjal bhartha',
             'cabbage and peas','pattagobhi aur matar','carrot and fenugreek','gajar methi',
             'beans with coconut','cauliflower with coconut','sarson ka saag','mustard greens',
             'mixed vegetable curry','mix veg','tofu','pea vadi curry','lotus stem curry','creamed spinach',
             'stuffed okra','bharwa bhindi','stuffed bottle gourd','stuffed ghiya',
             'peas brinjal','matar baingan','okra fry','bhindi sabzi','bhindi subji',
             'stuffed bittergourd','bharwa karela','spinach paneer','kofta curry'
             ],
    'curd': ['curd','dahi','yogurt','raita','chaas','buttermilk','shrikhand','mishti doi',
             'cucumber raita','mint raita','peanut raita','carrot and spinach raita',
             'tomato onion raita','grapes raita','bottle gourd raita','sprouted moong raita',
             'banana raita','sweet raita'],
    'protein_nonveg': [
             # Egg items
             'boiled egg','fried egg','poached egg','scrambled egg','stuffed egg omelette',
             'baked egg','deviled egg','egg curry','egg bhujia','indian style egg bhujia',
             # Chicken
             'chicken curry','butter chicken','tandoori chicken','chicken kebab','chilli chicken',
             'afghani chicken','handi chicken','lemon chicken','roast chicken','shahi chicken',
             'tomato chicken','creamy chicken','ginger chicken','chicken korma','chicken stew',
             'chicken manchurian','cajun chicken','fried chicken','broccoli chicken',
             'chicken and tomato','chicken sweet and sour',
             # Fish
             'fish curry','fried fish','tomato fish','baked fish','fish tikka','tandoori fish',
             'hariyali fish','lemon butter fish','bengal fish curry','fish in coconut milk',
             'crispy baked fish','fish finger','fish orly',
             # Seafood / other
             'prawn curry','baked stuffed fish',
             # Mutton/keema
             'spinach mutton','mutton do piaza','mutton korma','kashmiri mutton',
             'keema kofta curry','shahi keema kofta','minced meat ball curry',
             'nargisi kofta','pea keema curry'],
    'rice': ['boiled rice','plain rice','steamed rice','biryani','biriyani','pulao',
             'fried rice','pongal','curd rice','lemon rice','tamarind rice','khichdi','khichri'],
    'snack':['sandwich','toast','dhokla','khaman','pakora','pakoda','tikki',
             'upma','cutlet','chaat','seekh kebab','kebab','roll','pin wheel'],
    'breakfast_carb': ['oats','cornflakes','porridge','daliya','sheera','vermicelli upma',
                       'poha with curd','semolina idli','instant idli','idli','dosa','uttapam','appam']
}

def assign_role(name):
    n = name.lower()
    for role in ['roti','dal','sabzi','curd','protein_nonveg','snack','breakfast_carb','rice']:
        for kw in ROLE_KW[role]:
            if kw in n:
                if role == 'dal' and any(r in n for r in ['parantha','paratha','poori','puri']):
                    return 'roti'
                if kw == 'kofta curry' and classify_diet(name) == 'nonveg':
                    return 'protein_nonveg'
                return role
    return 'other'

def assign_grain(name):
    n = name.lower()
    if any(k in n for k in ['makki','bajra','jowar','ragi','millet','sorghum','maize porridge']): return 'millet'
    if any(k in n for k in ['oats','cornflakes','oatmeal']): return 'oats'
    if any(k in n for k in ['rice','biryani','biriyani','pulao','pongal','idli','dosa','appam','uttapam','khichdi','khichri']): return 'rice'
    if any(k in n for k in ['daliya','semolina','suji','rava','vermicelli','sheera']): return 'semolina'
    if any(k in n for k in ['roti','chapati','paratha','parantha','naan','thepla','laccha','puranpoli','bhatura','phulka','kulcha']): return 'wheat'
    return 'other'

def assign_subcat(row):
    name = row['food_name'].lower(); role = row['food_role']
    if role == 'dal':
        if any(k in name for k in ['moong']): return 'moong'
        if any(k in name for k in ['urad','moti mahal']): return 'urad'
        if any(k in name for k in ['masoor']): return 'masoor'
        if any(k in name for k in ['moth']): return 'moth'
        if any(k in name for k in ['rajma','rajmah','kidney']): return 'rajma'
        if any(k in name for k in ['chole','chhole','chickpeas','black channa']): return 'chana'
        if any(k in name for k in ['lobia']): return 'lobia'
        if any(k in name for k in ['soya','soyabean']): return 'soya'
        if any(k in name for k in ['arhar','toor']): return 'arhar'
        if any(k in name for k in ['sambar']): return 'sambar'
        return 'mixed'
    if role == 'sabzi':
        if any(k in name for k in ['paneer','tofu','soya']): return 'paneer'
        if any(k in name for k in ['aloo','potato','dum']): return 'potato'
        if any(k in name for k in ['palak','spinach','sarson','saag','methi']): return 'leafy'
        if any(k in name for k in ['bhindi','karela','lauki','tinda','parval','okra']): return 'gourd'
        if any(k in name for k in ['matar','peas','beans']): return 'legume'
        if any(k in name for k in ['baingan','brinjal']): return 'brinjal'
        return 'other_veg'
    if role == 'curd':
        if any(k in name for k in ['mint','pudina','cucumber','tomato','onion']): return 'savory'
        if any(k in name for k in ['peanut','moong','vegetable','salad']): return 'protein_raita'
        if any(k in name for k in ['banana','grapes','sweet']): return 'sweet'
        if any(k in name for k in ['chaas','buttermilk']): return 'liquid'
        if any(k in name for k in ['spinach','carrot','bottle gourd','ghiya']): return 'veggie'
        return 'plain'
    if role == 'snack':
        if any(k in name for k in ['sandwich','toast']): return 'sandwich'
        if any(k in name for k in ['dhokla','khaman']): return 'steamed'
        if any(k in name for k in ['upma','poha']): return 'cooked_grain'
        if any(k in name for k in ['pakora','pakoda','tikki']): return 'fried'
        if any(k in name for k in ['kebab','cutlet']): return 'protein'
        if any(k in name for k in ['salad','sprout','chat']): return 'salad'
        return 'other'
    if role == 'protein_nonveg':
        if any(k in name for k in NONVEG_KW): return 'meat'
        if any(k in name for k in EGG_KW): return 'egg'
        return 'protein'
    return 'other'

def classify_protein_subtype(name):
    n = name.lower()
    if any(k in n for k in NONVEG_KW):
        return 'meat'
    if any(k in n for k in EGG_KW):
        return 'egg'
    return 'na'

df['diet_type']      = df['food_name'].apply(classify_diet)
df['food_role']      = df['food_name'].apply(assign_role)
df['grain_category'] = df['food_name'].apply(assign_grain)
df['food_subcat']    = df.apply(assign_subcat, axis=1)
df['protein_subtype'] = df['food_name'].apply(classify_protein_subtype)
df.loc[df['food_role'] != 'protein_nonveg', 'protein_subtype'] = 'na'


## ── CELL 5 ── Glycemic + Composite Score

In [4]:
GROUP_PENALTY = {
    'refined': (['maida','white rice','refined','plain rice','boiled rice','bhatura','poori','puri','naan'], +0.8),
    'whole':   (['whole wheat','multigrain','oats','daliya','brown','whole moong','whole masoor','whole urad','whole moth','rajma','chole'], -0.5),
    'sugar':   (['sweet','meetha','meethi'], +1.0),
    'veg':     (['sabzi','bhaji','spinach','palak','gobhi','bhindi','karela','saag'], -0.4),
    'dairy':   (['curd','dahi','raita','yogurt','paneer'], -0.2),
}

def food_group_penalty(name):
    n, p = name.lower(), 0.0
    for _, (kws, val) in GROUP_PENALTY.items():
        if any(kw in n for kw in kws): p += val
    return p

df['gs_raw'] = 0.6*df['srv_carb'] + 1.8*df['srv_sugar'] - 1.2*df['srv_fibre'] - 0.8*df['srv_protein'] - 0.3*df['srv_fat']
df['gs_adj'] = df['gs_raw'] + df['food_name'].apply(food_group_penalty)
mu, sig = df['gs_adj'].mean(), df['gs_adj'].std() + 1e-9
df['gs_z']           = (df['gs_adj'] - mu) / sig
df['glycemic_score'] = 1 / (1 + np.exp(-df['gs_z']))

def norm(s): return (s - s.min()) / (s.max() - s.min() + 1e-9)

df['composite_score'] = (
    - 0.40 * norm(df['srv_carb'])
    - 0.30 * df['glycemic_score']
    + 0.20 * norm(df['srv_protein'])
    + 0.10 * norm(df['srv_fibre'])
    - 0.10 * norm(df['srv_sugar'])
)
print(f'GI score  : {df["glycemic_score"].min():.3f}–{df["glycemic_score"].max():.3f}')
print(f'Composite : {df["composite_score"].min():.3f}–{df["composite_score"].max():.3f}')

GI score  : 0.055–1.000
Composite : -0.783–0.180


## ── CELL 6 ── Pool Creation

In [5]:
SLOT_MAP = {
    'roti':           ['breakfast','lunch','dinner'],
    'dal':            ['lunch','dinner'],
    'sabzi':          ['lunch','dinner'],
    'curd':           ['breakfast','lunch','dinner','snack'],  # dinner: raita valid
    'protein_nonveg': ['breakfast','lunch','dinner','snack'],
    'rice':           ['lunch','dinner'],
    'snack':          ['breakfast','snack'],
    'breakfast_carb': ['breakfast'],
    'other':          [],
}
df['valid_slots'] = df['food_role'].map(SLOT_MAP)
pool = df.explode('valid_slots').dropna(subset=['valid_slots']).rename(columns={'valid_slots':'meal_slot'}).reset_index(drop=True)
print('Pool size per slot:'); print(pool['meal_slot'].value_counts())

Pool size per slot:
meal_slot
breakfast    196
lunch        179
dinner       179
snack        150
Name: count, dtype: int64


## ── CELL 7 ── Glucose Constraints

In [6]:
def glucose_status(g):
    if g < 100:  return 'Normal'
    if g < 126:  return 'Pre-diabetic'
    if g <= 180: return 'Diabetic'
    if g <= 250: return 'High glucose'
    return 'Very high glucose'

def get_constraints(glucose_mg_dl, daily_kcal):
    splits = {'breakfast':0.25,'lunch':0.35,'dinner':0.30,'snack':0.10}
    if glucose_mg_dl < 100:    carbs={'breakfast':70,'lunch':90,'dinner':80,'snack':35}; gs_base=0.72
    elif glucose_mg_dl < 126:  carbs={'breakfast':55,'lunch':70,'dinner':60,'snack':25}; gs_base=0.65
    elif glucose_mg_dl <= 180: carbs={'breakfast':40,'lunch':55,'dinner':45,'snack':20}; gs_base=0.60
    elif glucose_mg_dl <= 250: carbs={'breakfast':35,'lunch':49,'dinner':40,'snack':18}; gs_base=0.57
    else:                      carbs={'breakfast':30,'lunch':43,'dinner':35,'snack':15}; gs_base=0.54
    return {slot: {'kcal_target':round(daily_kcal*frac,1), 'carb_cap':carbs[slot],
                   'gs_cap':gs_base+(0.10 if slot=='breakfast' else 0.0),
                   'kcal_lo':round(daily_kcal*frac*0.88,1), 'kcal_hi':round(daily_kcal*frac*1.15,1)}
            for slot,frac in splits.items()}

print(f"{'Glucose':>6}  {'Status':<20} B   L   D   GI-cap")
for g in [90, 115, 150, 200, 260]:
    c = get_constraints(g, 2400)
    print(f"{g:6d}  {glucose_status(g):<20} {c['breakfast']['carb_cap']:2g}g {c['lunch']['carb_cap']:2g}g {c['dinner']['carb_cap']:2g}g  ≤{c['lunch']['gs_cap']:.2f}")

Glucose  Status               B   L   D   GI-cap
    90  Normal               70g 90g 80g  ≤0.72
   115  Pre-diabetic         55g 70g 60g  ≤0.65
   150  Diabetic             40g 55g 45g  ≤0.60
   200  High glucose         35g 49g 40g  ≤0.57
   260  Very high glucose    30g 43g 35g  ≤0.54


## ── CELL 8 ── Core Utilities (Anti-Repetition Engine)

In [7]:
OUTPUT_COLS = ['food_name','food_role','food_subcat','grain_category','serving_unit',
               'portions','kcal','carb_g','protein_g','fat_g','fibre_g','glycemic_score','composite_score']
MIN_SRV = {'roti':1,'dal':1,'sabzi':1,'curd':1,'protein_nonveg':1,'snack':1,'breakfast_carb':1,'rice':1,'other':1}
MAX_SRV = {'roti':3,'dal':2,'sabzi':2,'curd':1,'protein_nonveg':2,'snack':2,'breakfast_carb':3,'rice':2,'other':1}

TOP_N         = 5
FAMILY_CNT_P  = 0.15
RECENCY_P     = 0.45
GRAIN_P       = 0.20
COOLDOWN_N    = 4

VALID_RULES = {
    'breakfast': ([{'roti','rice','breakfast_carb','snack'}], 'needs carb'),
    'lunch':     ([{'roti','rice'}, {'dal','protein_nonveg'}, {'sabzi','curd','dal'}], 'carb+protein+veg'),
    'dinner':    ([{'roti','rice'}, {'dal','protein_nonveg','sabzi'}, {'sabzi','curd','dal'}], 'carb+(dal/sabzi)+veg'),
    'snack':     ([{'snack','curd','protein_nonveg'}], 'needs snack'),
}

def validate_meal(meal, slot):
    if meal.empty: return False
    roles = set(meal['food_role'])
    rules, _ = VALID_RULES.get(slot, ([set()], ''))
    return all(bool(roles & g) for g in rules)

def _make_item(row, role, portions):
    return {'food_name':row['food_name'], 'food_role':role, 'food_subcat':row.get('food_subcat','other'),
            'grain_category':row.get('grain_category','other'), 'serving_unit':row['servings_unit'],
            'portions':portions, 'kcal':round(row['srv_kcal']*portions,1), 'carb_g':round(row['srv_carb']*portions,1),
            'protein_g':round(row['srv_protein']*portions,1), 'fat_g':round(row['srv_fat']*portions,1),
            'fibre_g':round(row['srv_fibre']*portions,1), 'glycemic_score':round(row['glycemic_score'],3),
            'composite_score':round(row['composite_score'],3), '_unit_kcal':row['srv_kcal'], '_unit_carb':row['srv_carb']}

def _wu_add(week_used, role, food_name, food_subcat):
    if role not in week_used: week_used[role] = []
    if not isinstance(week_used[role], list): week_used[role] = []
    week_used[role].append((food_name, food_subcat))

def _diet_bonus(row, diet, role, slot):
    if role != 'protein_nonveg':
        return 0.0
    subtype = row.get('protein_subtype', 'na')
    if diet == 'egg':
        return 0.30 if subtype == 'egg' else -0.20
    if diet == 'nonveg':
        slot_bonus = 0.10 if slot in ('lunch', 'dinner') else 0.0
        return {'meat': 0.55 + slot_bonus, 'egg': 0.12, 'veg': -0.25}.get(subtype, 0.0)
    return 0.0

def pick_item(slot_pool, role, kcal_b, carb_b, gs_cap, day_used, week_used,
             fat_cap=None, avoid_grain=None, diet='veg', slot=None,
             required_subtype=None, banned_subtypes=None):
    role_hist      = week_used.get(role, [])
    used_names     = {t[0] for t in role_hist}
    cooldown_names = {t[0] for t in role_hist[-COOLDOWN_N:]}
    last4_subcats  = {t[1] for t in role_hist[-COOLDOWN_N:]}
    family_counts  = Counter(t[1] for t in role_hist)

    base = slot_pool[
        (slot_pool['food_role']      == role) &
        (slot_pool['glycemic_score'] <= gs_cap) &
        (slot_pool['srv_kcal']       <= kcal_b) &
        (slot_pool['srv_carb']       <= carb_b) &
        (~slot_pool['food_name'].isin(day_used))
    ].copy()

    if fat_cap is not None:
        base = base[base['srv_fat'] <= fat_cap]
    if required_subtype is not None and 'protein_subtype' in base.columns:
        base = base[base['protein_subtype'] == required_subtype]
    if banned_subtypes and 'protein_subtype' in base.columns:
        base = base[~base['protein_subtype'].isin(banned_subtypes)]
    if base.empty:
        return None

    fresh = base[~base['food_name'].isin(used_names)]
    if fresh.empty:
        fresh = base[~base['food_name'].isin(cooldown_names)]
        if fresh.empty:
            fresh = base

    fresh = fresh.copy()
    fresh['_score'] = fresh['composite_score'].copy()

    item_counts = Counter(t[0] for t in role_hist)
    if item_counts:
        fresh['_score'] -= fresh['food_name'].map(lambda n: item_counts.get(n, 0) * 0.25)

    if family_counts:
        fresh['_score'] -= fresh['food_subcat'].map(lambda s: family_counts.get(s, 0) * FAMILY_CNT_P)

    fresh.loc[fresh['food_subcat'].isin(last4_subcats), '_score'] -= RECENCY_P

    if avoid_grain:
        fresh.loc[fresh['grain_category'] == avoid_grain, '_score'] -= GRAIN_P

    fresh['_score'] += fresh.apply(lambda r: _diet_bonus(r, diet, role, slot), axis=1)

    top5 = fresh.nlargest(min(TOP_N, len(fresh)), '_score')
    return top5.sample(1).iloc[0]

def scale_portions(row, role, kcal_b, carb_b):
    max_srv = MAX_SRV.get(role, 1)
    if role == 'breakfast_carb' and row['srv_kcal'] < 200:
        max_srv = max(3, int(np.ceil(320 / row['srv_kcal'])))
    mk = int(kcal_b // row['srv_kcal'])
    mc = int(carb_b // row['srv_carb']) if row['srv_carb'] > 0 else max_srv
    p  = max(MIN_SRV.get(role,1), min(mk, mc, max_srv))
    return _make_item(row, role, max(p, 1))

def _to_df(items):
    if not items: return pd.DataFrame(columns=OUTPUT_COLS)
    return pd.DataFrame(items)[OUTPUT_COLS]

def _add(items, row, role, kcal_b, carb_b, day_used, week_used):
    it = scale_portions(row, role, kcal_b, carb_b)
    items.append(it)
    day_used.add(row['food_name'])
    _wu_add(week_used, role, row['food_name'], row.get('food_subcat','other'))
    return it['kcal'], it['carb_g']

print('✅ Core utilities updated with diet-aware protein preference')


✅ Core utilities updated with diet-aware protein preference


## ── CELL 9 ── Calorie Optimizer

In [8]:
def optimize_calories(items, spool, C, day_used, week_used, pad_roles=None):
    if pad_roles is None: pad_roles = ['roti','dal','curd','snack']
    lo = C['kcal_lo']; hi = C['kcal_hi']; gsc = C['gs_cap']
    def total():  return sum(i['kcal']   for i in items)
    def c_used(): return sum(i['carb_g'] for i in items)

    for _ in range(3):  # Pass 1: trim
        if total() <= hi: break
        for it in items:
            if it['food_role'] in ['roti','rice'] and it['portions'] > 1:
                it['portions'] -= 1; it['kcal'] -= it['_unit_kcal']; it['carb_g'] -= it['_unit_carb']; break

    for it in items:    # Pass 2: scale up
        if total() >= lo: break
        if it['food_role'] in ['roti','rice']:
            gap = lo - total(); cleft = C['carb_cap'] - c_used()
            mx  = MAX_SRV[it['food_role']] - it['portions']
            by_k = int(gap // it['_unit_kcal']) + 1
            by_c = int(cleft // it['_unit_carb']) if it['_unit_carb'] > 0 else mx
            extra = min(by_k, by_c, mx)
            if extra > 0:
                it['portions'] += extra; it['kcal'] += round(it['_unit_kcal']*extra,1); it['carb_g'] += round(it['_unit_carb']*extra,1)

    if total() < lo:    # Pass 3a: add side item
        used_roles = {i['food_role'] for i in items}
        for role in pad_roles:
            if role in used_roles: continue
            gap = lo - total(); cleft = C['carb_cap'] - c_used()
            row = pick_item(spool, role, gap*1.8, cleft, gsc, day_used, week_used)
            if row is not None:
                it = scale_portions(row, role, gap*1.8, cleft); items.append(it)
                day_used.add(row['food_name']); _wu_add(week_used, role, row['food_name'], row.get('food_subcat','other'))
                break

    for it in items:    # Pass 3b: boost low-cal items
        if total() >= lo: break
        role = it['food_role']
        if role in ['breakfast_carb','dal','sabzi']:
            max_p = MAX_SRV.get(role,1)
            if role == 'breakfast_carb' and it['_unit_kcal'] < 200: max_p = 5
            if it['portions'] < max_p:
                it['portions'] += 1; it['kcal'] += round(it['_unit_kcal'],1); it['carb_g'] += round(it['_unit_carb'],1)

    return items

print('✅ optimize_calories() defined')

✅ optimize_calories() defined


## ── CELL 10 ── Meal Builders

In [9]:
def optimize_calories(items, spool, C, day_used, week_used, diet='veg', slot=None, pad_roles=None):
    if pad_roles is None: pad_roles = ['roti','dal','curd','snack']
    lo = C['kcal_lo']; hi = C['kcal_hi']; gsc = C['gs_cap']
    def total():  return sum(i['kcal']   for i in items)
    def c_used(): return sum(i['carb_g'] for i in items)

    for _ in range(3):
        if total() <= hi: break
        for it in items:
            if it['food_role'] in ['roti','rice'] and it['portions'] > 1:
                it['portions'] -= 1; it['kcal'] -= it['_unit_kcal']; it['carb_g'] -= it['_unit_carb']; break

    for it in items:
        if total() >= lo: break
        if it['food_role'] in ['roti','rice']:
            gap = lo - total(); cleft = C['carb_cap'] - c_used()
            mx  = MAX_SRV[it['food_role']] - it['portions']
            by_k = int(gap // it['_unit_kcal']) + 1
            by_c = int(cleft // it['_unit_carb']) if it['_unit_carb'] > 0 else mx
            extra = min(by_k, by_c, mx)
            if extra > 0:
                it['portions'] += extra; it['kcal'] += round(it['_unit_kcal']*extra,1); it['carb_g'] += round(it['_unit_carb']*extra,1)

    if total() < lo:
        used_roles = {i['food_role'] for i in items}
        for role in pad_roles:
            if role in used_roles: continue
            gap = lo - total(); cleft = C['carb_cap'] - c_used()
            row = pick_item(spool, role, gap*1.8, cleft, gsc, day_used, week_used, diet=diet, slot=slot)
            if row is not None:
                it = scale_portions(row, role, gap*1.8, cleft); items.append(it)
                day_used.add(row['food_name']); _wu_add(week_used, role, row['food_name'], row.get('food_subcat','other'))
                break

    for it in items:
        if total() >= lo: break
        role = it['food_role']
        if role in ['breakfast_carb','dal','sabzi','protein_nonveg']:
            max_p = MAX_SRV.get(role,1)
            if role == 'breakfast_carb' and it['_unit_kcal'] < 200: max_p = 5
            if it['portions'] < max_p:
                it['portions'] += 1; it['kcal'] += round(it['_unit_kcal'],1); it['carb_g'] += round(it['_unit_carb'],1)

    return items

def _count_meat_items(week_used):
    return sum(1 for _, subcat in week_used.get('protein_nonveg', []) if subcat == 'meat')

def build_breakfast(pool, C, day_used, week_used, diet, day_state):
    spool = diet_filter(pool[pool['meal_slot']=='breakfast'], diet)
    Cc = C['breakfast']; gs = Cc['gs_cap']; items = []; ku = cu = 0.0
    for role in ['breakfast_carb','roti']:
        row = pick_item(spool, role, Cc['kcal_target']*0.72, Cc['carb_cap'], gs, day_used, week_used, diet=diet, slot='breakfast')
        if row is not None:
            dk, dc = _add(items, row, role, Cc['kcal_target']*0.72, Cc['carb_cap'], day_used, week_used)
            ku += dk; cu += dc; day_state['last_grain'] = row.get('grain_category','other'); break
    if ku < Cc['kcal_target']*0.40:
        fill_kcal = Cc['kcal_target']*0.72 - ku; fill_carb = Cc['carb_cap'] - cu
        for r2 in ['snack','roti','breakfast_carb']:
            row2 = pick_item(spool, r2, fill_kcal, fill_carb, gs, day_used, week_used, diet=diet, slot='breakfast')
            if row2 is not None:
                dk, dc = _add(items, row2, r2, fill_kcal, fill_carb, day_used, week_used)
                ku += dk; cu += dc; break
    if diet in ['egg','nonveg']:
        subtype = 'egg' if diet == 'egg' else None
        row = pick_item(spool, 'protein_nonveg', Cc['kcal_target']-ku, Cc['carb_cap']-cu, gs, day_used, week_used,
                        diet=diet, slot='breakfast', required_subtype=subtype)
        if row is not None:
            dk, dc = _add(items, row, 'protein_nonveg', Cc['kcal_target']-ku, Cc['carb_cap']-cu, day_used, week_used)
            ku += dk; cu += dc
    ck = Cc['kcal_target']-ku; cc = Cc['carb_cap']-cu
    if ck > 60:
        row = pick_item(spool, 'curd', ck, cc, gs, day_used, week_used, diet=diet, slot='breakfast')
        if row is not None: _add(items, row, 'curd', ck, cc, day_used, week_used)
    items = optimize_calories(items, spool, Cc, day_used, week_used, diet=diet, slot='breakfast',
                              pad_roles=['roti','breakfast_carb','snack','curd'])
    return _to_df(items)


def build_thali(pool, slot, C, day_used, week_used, diet, lighter=False, avoid_grain=None, force_meat=False):
    spool = diet_filter(pool[pool['meal_slot']==slot], diet)
    Cc = C[slot]; gs = Cc['gs_cap']; items = []; ku = cu = 0.0
    roti_min = Cc['kcal_target'] * 0.25
    dal_b    = Cc['kcal_target'] * (0.32 if diet == 'nonveg' else 0.40)
    sabzi_b  = Cc['kcal_target'] * 0.18
    protein_b = Cc['kcal_target'] * (0.34 if diet == 'nonveg' else 0.22)
    cd = Cc['carb_cap'] * (0.40 if diet == 'nonveg' else 0.65)
    cs = Cc['carb_cap'] * 0.22
    cp = Cc['carb_cap'] * 0.30

    dal_placed = False
    protein_placed = False

    if diet in ['egg', 'nonveg']:
        subtype = 'meat' if (diet == 'nonveg' and force_meat) else ('egg' if diet == 'egg' else None)
        banned = ['egg'] if (diet == 'nonveg' and force_meat) else None
        for km in [1.0, 1.25, 1.5]:
            row = pick_item(spool, 'protein_nonveg', protein_b*km, cp, gs, day_used, week_used,
                            diet=diet, slot=slot, required_subtype=subtype, banned_subtypes=banned)
            if row is not None:
                dk, dc = _add(items, row, 'protein_nonveg', protein_b*km, cp, day_used, week_used)
                ku += dk; cu += dc; protein_placed = True; break

    should_add_dal = True
    if diet == 'nonveg' and protein_placed:
        kcal_left = Cc['kcal_target'] - ku
        carb_left = Cc['carb_cap'] - cu
        should_add_dal = (slot == 'lunch' and kcal_left >= Cc['kcal_target'] * 0.22 and carb_left >= Cc['carb_cap'] * 0.18)

    if should_add_dal:
        for cm in [1.0, 1.6, 2.0]:
            row = pick_item(spool, 'dal', dal_b, cd*cm, gs, day_used, week_used, diet=diet, slot=slot)
            if row is not None:
                dk, dc = _add(items, row, 'dal', dal_b, cd*cm, day_used, week_used)
                ku += dk; cu += dc; dal_placed = True; break

    for cm in [1.0, 1.5, 2.0]:
        row = pick_item(spool, 'sabzi', sabzi_b, cs*cm, gs, day_used, week_used, diet=diet, slot=slot)
        if row is not None:
            dk, dc = _add(items, row, 'sabzi', sabzi_b, cs*cm, day_used, week_used)
            ku += dk; cu += dc; break

    roti_b = max(Cc['kcal_target']-ku-Cc['kcal_target']*0.06, roti_min)
    carb_r = Cc['carb_cap'] - cu
    row = pick_item(spool, 'roti', roti_b, carb_r, gs, day_used, week_used, avoid_grain=avoid_grain, diet=diet, slot=slot)
    if row is not None:
        dk, dc = _add(items, row, 'roti', roti_b, carb_r, day_used, week_used)
        ku += dk; cu += dc

    ck = Cc['kcal_target']-ku; cc = Cc['carb_cap']-cu
    if diet != 'nonveg' or not protein_placed:
        row = pick_item(spool, 'curd', ck, cc, gs, day_used, week_used, diet=diet, slot=slot)
        if row is not None:
            dk, dc = _add(items, row, 'curd', ck, cc, day_used, week_used)
            ku += dk; cu += dc

    if diet == 'nonveg' and not protein_placed:
        row = pick_item(spool, 'protein_nonveg', Cc['kcal_target']-ku, Cc['carb_cap']-cu, gs, day_used, week_used,
                        diet=diet, slot=slot, required_subtype='meat' if force_meat else None)
        if row is not None:
            dk, dc = _add(items, row, 'protein_nonveg', Cc['kcal_target']-ku, Cc['carb_cap']-cu, day_used, week_used)
            ku += dk; cu += dc

    allow_protein = (not lighter) or (lighter and not dal_placed and not protein_placed)
    if allow_protein and diet == 'egg' and not protein_placed:
        row = pick_item(spool, 'protein_nonveg', Cc['kcal_target']-ku, Cc['carb_cap']-cu, gs, day_used, week_used,
                        diet=diet, slot=slot, required_subtype='egg')
        if row is not None:
            dk, dc = _add(items, row, 'protein_nonveg', Cc['kcal_target']-ku, Cc['carb_cap']-cu, day_used, week_used)
            ku += dk; cu += dc

    pad_roles = ['roti','sabzi','protein_nonveg'] if diet == 'nonveg' else ['roti','dal','sabzi','curd']
    items = optimize_calories(items, spool, Cc, day_used, week_used, diet=diet, slot=slot, pad_roles=pad_roles)
    return _to_df(items)


def build_snack(pool, C, day_used, week_used, diet):
    spool = diet_filter(pool[pool['meal_slot']=='snack'], diet)
    Cc = C['snack']; gs = Cc['gs_cap']; budget = min(Cc['kcal_target'], 300); items = []; ku = 0.0
    for _ in range(2):
        if ku >= budget*0.85: break
        roles = ['snack', 'curd']
        if diet == 'egg':
            roles.append('protein_nonveg')
        elif diet == 'nonveg':
            roles = ['protein_nonveg', 'snack', 'curd']
        for role in roles:
            req = 'egg' if (diet == 'egg' and role == 'protein_nonveg') else None
            row = pick_item(spool, role, budget-ku, Cc['carb_cap'], gs, day_used, week_used,
                            fat_cap=12.0, diet=diet, slot='snack', required_subtype=req)
            if row is not None:
                dk, _ = _add(items, row, role, budget-ku, Cc['carb_cap'], day_used, week_used)
                ku += dk; break
    return _to_df(items)

print('✅ Meal builders updated for stronger non-veg allocation')


✅ Meal builders updated for stronger non-veg allocation


## ── CELL 11 ── Daily Plan Generator

In [10]:
def generate_daily_plan(glucose_mg_dl=130, daily_kcal=2400, diet='veg', seed=None, week_used=None):
    """
    Generate one day's meal plan.
    Pass week_used back into every subsequent call for multi-day variety.
    week_used: dict[str, list[tuple[food_name, food_subcat]]]
    """
    assert diet in ('veg','egg','nonveg'), "diet must be 'veg', 'egg', or 'nonveg'"
    if seed is not None: random.seed(seed); np.random.seed(seed)
    if week_used is None: week_used = {}
    C = get_constraints(glucose_mg_dl, daily_kcal)
    day_used = set(); day_state = {'last_grain': None}

    meat_so_far = _count_meat_items(week_used)
    force_lunch_meat = (diet == 'nonveg' and meat_so_far < 4)
    force_dinner_meat = (diet == 'nonveg' and meat_so_far < 7)

    breakfast = build_breakfast(pool, C, day_used, week_used, diet, day_state)
    if not breakfast.empty:
        cr = breakfast[breakfast['food_role'].isin(['roti','breakfast_carb','rice'])]
        if not cr.empty: day_state['last_grain'] = cr.iloc[0]['grain_category']

    lunch = build_thali(pool, 'lunch', C, day_used, week_used, diet,
                        avoid_grain=day_state.get('last_grain'), force_meat=force_lunch_meat)
    if not lunch.empty:
        rr = lunch[lunch['food_role']=='roti']
        if not rr.empty: day_state['last_grain'] = rr.iloc[0]['grain_category']

    dinner = build_thali(pool, 'dinner', C, day_used, week_used, diet,
                         lighter=True, avoid_grain=day_state.get('last_grain'),
                         force_meat=force_dinner_meat)
    snack  = build_snack(pool, C, day_used, week_used, diet)

    all_items = pd.concat([breakfast, lunch, dinner, snack])
    meal_kcal = {s: round(m['kcal'].sum(),1) if not m.empty else 0.0
                 for s,m in zip(['breakfast','lunch','dinner','snack'],[breakfast,lunch,dinner,snack])}
    summary = {
        'glucose_mg_dl':glucose_mg_dl, 'status':glucose_status(glucose_mg_dl), 'diet':diet,
        'daily_kcal_target':daily_kcal, 'total_kcal':round(all_items['kcal'].sum(),1),
        'total_carb_g':round(all_items['carb_g'].sum(),1), 'total_protein_g':round(all_items['protein_g'].sum(),1),
        'total_fat_g':round(all_items['fat_g'].sum(),1), 'total_fibre_g':round(all_items['fibre_g'].sum(),1),
        'meal_kcal':meal_kcal, 'kcal_targets':{s:C[s]['kcal_target'] for s in C},
        'pct_achieved':round(all_items['kcal'].sum()/daily_kcal*100, 1),
        'meat_items_week': _count_meat_items(week_used),
    }
    return dict(breakfast=breakfast, lunch=lunch, dinner=dinner, snack=snack, summary=summary, week_used=week_used)

print('✅ generate_daily_plan() updated with weekly meat quota support')


✅ generate_daily_plan() updated with weekly meat quota support


## ── CELL 12 ── Display

In [11]:
EMOJIS = {'breakfast':'🌅  BREAKFAST','lunch':'☀️  LUNCH','dinner':'🌙  DINNER','snack':'🍎  SNACK'}
FMT = {'portions':'{:.0f}','kcal':'{:.0f}','carb_g':'{:.1f}','protein_g':'{:.1f}',
       'fat_g':'{:.1f}','fibre_g':'{:.1f}','glycemic_score':'{:.3f}','composite_score':'{:.3f}'}

def display_plan(plan, show_scores=True):
    s = plan['summary']; pct = s['pct_achieved']
    bar = '█'*int(pct//5)+'░'*(20-int(pct//5))
    print('='*72); print('  HealthSYNQ-D v8  ·  FINAL')
    print(f"  Glucose : {s['glucose_mg_dl']} mg/dL  [{s['status']}]")
    print(f"  Diet    : {s['diet'].upper()}")
    print(f"  Target  : {s['daily_kcal_target']:.0f} kcal  Generated: {s['total_kcal']:.0f} kcal  ({pct:.1f}%)")
    print(f"  [{bar}]")
    print('='*72)
    show_cols = OUTPUT_COLS if show_scores else [c for c in OUTPUT_COLS if c not in ('glycemic_score','composite_score')]
    for slot in ['breakfast','lunch','dinner','snack']:
        meal = plan[slot]; got = s['meal_kcal'][slot]; tgt = s['kcal_targets'][slot]
        valid = validate_meal(meal, slot)
        print(f"\n{'─'*72}")
        print(f"  {EMOJIS[slot]}")
        print(f"  Target {tgt:.0f} kcal → {got:.0f} kcal ({got/tgt*100:.0f}%)  {'✅ VALID' if valid else '⚠️ INCOMPLETE'}")
        print(f"{'─'*72}")
        if meal.empty: print('  ⚠️  No items')
        else: display(meal[show_cols].style.format({k:v for k,v in FMT.items() if k in show_cols}).set_properties(**{'text-align':'left'}).hide(axis='index'))
    print(f"\n{'='*72}"); print('  📊 DAILY TOTALS')
    print(f"  Energy: {s['total_kcal']:.0f}/{s['daily_kcal_target']:.0f} kcal ({pct:.1f}%)  |  Carbs: {s['total_carb_g']:.1f}g  |  Protein: {s['total_protein_g']:.1f}g  |  Fat: {s['total_fat_g']:.1f}g")
    print('='*72)

print('✅ display_plan() defined')

✅ display_plan() defined


## ── CELL 13 ── 🚀 Single Day Run

In [12]:
# ╔══════════════════════════════════════════════════╗
GLUCOSE_MG_DL = 130      # fasting glucose (mg/dL)
DAILY_KCAL    = 2400     # calorie target
DIET          = 'veg'    # 'veg' | 'egg' | 'nonveg'
SEED          = None     # int = reproducible, None = fresh random
# ╚══════════════════════════════════════════════════╝

plan = generate_daily_plan(glucose_mg_dl=GLUCOSE_MG_DL, daily_kcal=DAILY_KCAL, diet=DIET, seed=SEED)
display_plan(plan)

  HealthSYNQ-D v8  ·  FINAL
  Glucose : 130 mg/dL  [Diabetic]
  Diet    : VEG
  Target  : 2400 kcal  Generated: 2724 kcal  (113.5%)
  [██████████████████████]

────────────────────────────────────────────────────────────────────────
  🌅  BREAKFAST
  Target 600 kcal → 924 kcal (154%)  ✅ VALID
────────────────────────────────────────────────────────────────────────


food_name,food_role,food_subcat,grain_category,serving_unit,portions,kcal,carb_g,protein_g,fat_g,fibre_g,glycemic_score,composite_score
Appam,breakfast_carb,other,rice,appam,2,819,40.6,4.8,34.6,9.5,0.372,-0.123
Cucumber and yogurt salad (Kheere aur dahi ka salad),curd,savory,other,small plate,1,42,5.4,2.4,1.1,1.9,0.401,-0.124
Paushtik roti,roti,other,wheat,roti,1,64,11.4,3.1,0.6,2.5,0.398,-0.132



────────────────────────────────────────────────────────────────────────
  ☀️  LUNCH
  Target 840 kcal → 772 kcal (92%)  ✅ VALID
────────────────────────────────────────────────────────────────────────


food_name,food_role,food_subcat,grain_category,serving_unit,portions,kcal,carb_g,protein_g,fat_g,fibre_g,glycemic_score,composite_score
Whole urad (Urad ki dal),dal,urad,other,bowl,2,365,34.2,7.5,8.9,7.9,0.356,-0.108
Carrot and fenugreek leaves (Gajar methi),sabzi,leafy,other,bowl,2,236,16.0,4.0,7.3,8.4,0.363,-0.096
Chapati/Roti,roti,other,wheat,chapati,2,146,25.7,4.2,2.6,4.5,0.403,-0.139
Curd mint dip,curd,savory,other,tablespoon,1,25,3.0,1.6,0.8,0.3,0.408,-0.127



────────────────────────────────────────────────────────────────────────
  🌙  DINNER
  Target 720 kcal → 791 kcal (110%)  ✅ VALID
────────────────────────────────────────────────────────────────────────


food_name,food_role,food_subcat,grain_category,serving_unit,portions,kcal,carb_g,protein_g,fat_g,fibre_g,glycemic_score,composite_score
Soyabean curry,dal,soya,other,bowl,2,392,16.2,12.5,12.2,8.8,0.310,-0.062
Bottle gourd soup (Ghiya/Lauki soup),sabzi,gourd,other,bowl,1,68,3.4,1.3,5.3,2.8,0.372,-0.109
Makki ki roti,roti,other,millet,roti,1,224,20.6,3.0,14.3,4.4,0.391,-0.145
Spinach raita (Palak ka raita),curd,veggie,other,bowl,1,107,10.3,8.0,3.8,2.8,0.435,-0.134



────────────────────────────────────────────────────────────────────────
  🍎  SNACK
  Target 240 kcal → 237 kcal (99%)  ✅ VALID
────────────────────────────────────────────────────────────────────────


food_name,food_role,food_subcat,grain_category,serving_unit,portions,kcal,carb_g,protein_g,fat_g,fibre_g,glycemic_score,composite_score
Cabbage rolls (dry) ((Pattagobhi rolls) (dry)),snack,other,other,roll,2,237,10.1,13.6,15.6,3.1,0.355,-0.100



  📊 DAILY TOTALS
  Energy: 2724/2400 kcal (113.5%)  |  Carbs: 196.9g  |  Protein: 66.0g  |  Fat: 107.1g


## ── CELL 14 ── 📅 Multi-Day Plan

In [13]:
# ╔══════════════════════════════════════╗
N_DAYS        = 7
GLUCOSE_MG_DL = 130
DAILY_KCAL    = 2400
DIET          = 'veg'
# ╚══════════════════════════════════════╝

week_used = {}; all_foods = []; daily_pcts = []; grain_log = []

for day in range(1, N_DAYS + 1):
    plan = generate_daily_plan(glucose_mg_dl=GLUCOSE_MG_DL, daily_kcal=DAILY_KCAL, diet=DIET, week_used=week_used)
    week_used = plan['week_used']; s = plan['summary']; daily_pcts.append(s['pct_achieved'])
    print(f"\n{'='*65}\n  DAY {day}  —  {s['total_kcal']:.0f}/{s['daily_kcal_target']:.0f} kcal ({s['pct_achieved']:.1f}%)\n{'='*65}")
    for slot in ['breakfast','lunch','dinner','snack']:
        meal = plan[slot]; got = s['meal_kcal'][slot]; tgt = s['kcal_targets'][slot]
        valid = validate_meal(meal, slot)
        carb_r = meal[meal['food_role'].isin(['roti','rice','breakfast_carb'])]
        grain  = carb_r.iloc[0]['grain_category'] if not carb_r.empty else '-'
        if slot != 'snack': grain_log.append(grain)
        foods = ' | '.join(meal['food_name'].str[:18].tolist()) if not meal.empty else 'EMPTY'
        print(f"  {'✅' if valid else '⚠️'} {slot:10s} {got:5.0f}/{tgt:5.0f}  [{grain:8s}]  {foods}")
        if not meal.empty: all_foods.extend(meal['food_name'].tolist())

counts = Counter(all_foods); repeated = {k:v for k,v in counts.items() if v>1}
dal_seq = [t[1] for t in week_used.get('dal', [])]
sab_seq = [t[1] for t in week_used.get('sabzi', [])]
print(f"\n{'─'*65}\n  📊 {N_DAYS}-DAY REPORT  ({DIET})")
print(f"  Foods    : {len(all_foods)} instances, {len(counts)} unique, {len(repeated)} repeated")
print(f"  Avg kcal : {sum(daily_pcts)/len(daily_pcts):.1f}% of target")
print(f"  Dal fam  : {dal_seq}")
print(f"  Sabzi cat: {sab_seq[:10]}")
print(f"  Grain seq: {grain_log[:9]}")
for k,v in sorted(repeated.items(),key=lambda x:-x[1])[:6]: print(f"    {k[:52]:52s}: {v}x")


  DAY 1  —  2581/2400 kcal (107.6%)
  ✅ breakfast    916/  600  [rice    ]  Appam | Curd mint dip | Chapati/Roti
  ✅ lunch        833/  840  [wheat   ]  Mushroom matar | Stuffed okra (Bhar | Besan and spinach 
  ✅ dinner       635/  720  [wheat   ]  Whole moong (Moong | Sarson ka saag | Paushtik roti | Green chilli raita
  ✅ snack        197/  240  [-       ]  Cabbage rolls (cur | Curd vegetable dip

  DAY 2  —  2065/2400 kcal (86.0%)
  ✅ breakfast    458/  600  [semolina]  Lentils and wheat  | Pin wheel sandwich | Curd vegetable dip
  ✅ lunch        758/  840  [wheat   ]  Black channa curry | Bottle gourd raita | Pea parantha/parat | Cucumber and yogur
  ✅ dinner       644/  720  [wheat   ]  Whole masoor (Maso | Bottle gourd soup  | Cauliflower parant | Curd mint dip
  ✅ snack        205/  240  [-       ]  Dhokla | Cabbage rolls (dry

  DAY 3  —  1989/2400 kcal (82.9%)
  ✅ breakfast    259/  600  [rice    ]  Idli | Pin wheel sandwich
  ✅ lunch        868/  840  [wheat   ]  Moti mahal

## ── CELL 15 ── Compare 3 Diet Modes

In [14]:
for diet_mode in ['veg','egg','nonveg']:
    p = generate_daily_plan(glucose_mg_dl=130, daily_kcal=2400, diet=diet_mode, seed=42)
    s = p['summary']
    print(f"\n{'='*60}\n  {diet_mode.upper():8s}  {s['total_kcal']:.0f}/{s['daily_kcal_target']:.0f} kcal ({s['pct_achieved']:.1f}%)")
    for slot in ['breakfast','lunch','dinner','snack']:
        meal = p[slot]; got = s['meal_kcal'][slot]; tgt = s['kcal_targets'][slot]
        foods = ' | '.join(meal['food_name'].str[:20].tolist()) if not meal.empty else 'EMPTY'
        print(f"  {'✅' if validate_meal(meal,slot) else '⚠️'} {slot:10s} {got:5.0f}/{tgt:5.0f}  {foods}")


  VEG       2216/2400 kcal (92.3%)
  ✅ breakfast    421/  600  Moong bean dosa (Pes | Cucumber and yogurt 
  ✅ lunch        753/  840  Whole urad (Urad ki  | Bottle gourd soup (G | Methi thepla
  ✅ dinner       804/  720  Soyabean curry | Sarson ka saag | Besan and spinach pa | Bathua raita
  ✅ snack        237/  240  Cabbage rolls (dry) 

  EGG       2403/2400 kcal (100.1%)
  ✅ breakfast    558/  600  Moong bean dosa (Pes | Boiled egg (Ubla and | Cucumber and yogurt 
  ✅ lunch        849/  840  Scrambled egg (Ande  | Whole urad (Urad ki  | Stuffed okra (Bharwa | Chapati/Roti | Curd vegetable dip
  ✅ dinner       757/  720  Deviled egg | Soyabean curry | Carrot and fenugreek | Paushtik roti
  ✅ snack        239/  240  Pin wheel sandwich | Khaman (dhokla)

  NONVEG    2303/2400 kcal (96.0%)
  ✅ breakfast    528/  600  Moong bean dosa (Pes | Tandoori fish
  ✅ lunch        821/  840  Hariyali Fish Tikka | Soyabean curry | Stuffed okra (Bharwa | Methi thepla
  ✅ dinner       742/  720  To

## ── CELL 16 ── Inspect Food Pool

In [15]:
for role in ['roti','dal','sabzi','curd','protein_nonveg','snack','breakfast_carb','rice']:
    sub = df[df['food_role']==role][['food_name','diet_type','food_subcat','grain_category','servings_unit','srv_kcal','glycemic_score','composite_score']].sort_values('composite_score',ascending=False)
    print(f"\n=== {role.upper()} ({len(sub)} items) ===")
    print(sub.head(6).to_string(index=False))


=== ROTI (20 items) ===
                                                               food_name diet_type food_subcat grain_category servings_unit  srv_kcal  glycemic_score  composite_score
                                                  Keema parantha/paratha    nonveg       other          wheat      parantha     238.2             0.4             -0.1
                                                           Paushtik roti       veg       other          wheat          roti      64.1             0.4             -0.1
Besan and spinach parantha/paratha (Besan aur palak ka parantha/paratha)       veg       other          wheat      parantha     178.1             0.4             -0.1
                                                            Chapati/Roti       veg       other          wheat       chapati      72.8             0.4             -0.1
           Cauliflower parantha/paratha (Phoolgobhi ka parantha/paratha)       veg       other          wheat      parantha     181.4       

## ── CELL 17 ── Export to Excel

In [ ]:
def export_plan(plan, filename='healthsynqd_plan.xlsx'):
    with pd.ExcelWriter(filename, engine='openpyxl') as w:
        for slot in ['breakfast','lunch','dinner','snack']:
            plan[slot].to_excel(w, sheet_name=slot.capitalize(), index=False)
        pd.DataFrame([plan['summary']]).to_excel(w, sheet_name='Summary', index=False)
    print(f'✅ Exported → {filename}')

# export_plan(plan)

In [16]:
# ╔══════════════════════════════════════╗
N_DAYS     = 7
DAILY_KCAL = 2000
DIET       = 'veg'   # 🔁 change to 'nonveg' / 'mixed' later
# ╚══════════════════════════════════════╝

import random
from collections import Counter

# realistic glucose fluctuation (can tweak)
def generate_glucose_series(n):
    base = 240
    return [max(100, int(base + random.randint(-40, 60))) for _ in range(n)]

glucose_series = generate_glucose_series(N_DAYS)

week_used  = {}
all_foods  = []
daily_pcts = []
grain_log  = []

print(f"\n🧪 TEST RUN: {DIET.upper()} DIET | {N_DAYS} DAYS\n")

for day in range(N_DAYS):
    
    glucose = glucose_series[day]

    plan = generate_daily_plan(
        glucose_mg_dl = glucose,
        daily_kcal    = DAILY_KCAL,
        diet          = DIET,
        week_used     = week_used,
        seed          = None
    )

    week_used = plan['week_used']
    s         = plan['summary']
    daily_pcts.append(s['pct_achieved'])

    print(f"\n{'='*70}")
    print(f" DAY {day+1}  |  GLUCOSE: {glucose} mg/dL")
    print(f" KCAL: {s['total_kcal']:.0f}/{s['daily_kcal_target']:.0f} ({s['pct_achieved']:.1f}%)")
    print(f"{'='*70}")

    for slot in ['breakfast','lunch','dinner','snack']:
        meal  = plan[slot]
        got   = s['meal_kcal'][slot]
        tgt   = s['kcal_targets'][slot]
        valid = validate_meal(meal, slot)
        badge = '✅' if valid else '⚠️'

        # grain tracking
        carb_row = meal[meal['food_role'].isin(
            ['roti','rice','breakfast_carb'])] if not meal.empty else None
        
        grain = carb_row.iloc[0]['grain_category'] if carb_row is not None and not carb_row.empty else '-'
        
        if slot in ['breakfast','lunch','dinner']:
            grain_log.append(grain)

        foods = ' | '.join(
            meal['food_name'].str[:22].tolist()) if not meal.empty else 'EMPTY'

        print(f" {badge} {slot:10s} {got:4.0f}/{tgt:4.0f}  [{grain:8s}]  {foods}")

        if not meal.empty:
            all_foods.extend(meal['food_name'].tolist())

# ─────────────── REPORT ───────────────

counts   = Counter(all_foods)
repeated = {k: v for k, v in counts.items() if v > 1}

print(f"\n{'─'*70}")
print(f" 📊 WEEKLY REPORT ({DIET})")
print(f" Total meals        : {len(all_foods)}")
print(f" Unique foods       : {len(counts)}")
print(f" Repeated foods     : {len(repeated)}")
print(f" Avg kcal achieved  : {sum(daily_pcts)/len(daily_pcts):.1f}%")

if repeated:
    print("\n 🔁 Top Repeated Foods:")
    for k, v in sorted(repeated.items(), key=lambda x: -x[1])[:8]:
        print(f"   {k[:50]:50s} : {v}x")
else:
    print("\n ✅ No repetition detected")

print(f"\n 🌾 Grain Rotation:")
print("   " + " → ".join(grain_log[:12]) + " ...")


🧪 TEST RUN: VEG DIET | 7 DAYS


 DAY 1  |  GLUCOSE: 281 mg/dL
 KCAL: 1607/2000 (80.4%)
 ✅ breakfast   188/ 500  [rice    ]  Idli | Curd vegetable dip
 ✅ lunch       636/ 700  [wheat   ]  Black channa curry/Ben | Bottle gourd raita (Gh | Methi thepla
 ✅ dinner      588/ 600  [wheat   ]  Soyabean curry | Bottle gourd soup (Ghi | Paushtik roti
 ✅ snack       196/ 200  [-       ]  Khaman (dhokla) | Soya seekh kebab

 DAY 2  |  GLUCOSE: 214 mg/dL
 KCAL: 1892/2000 (94.6%)
 ✅ breakfast   380/ 500  [rice    ]  Moong bean dosa (Pesar
 ✅ lunch       703/ 700  [wheat   ]  Whole urad (Urad ki da | Carrot and fenugreek l | Besan and spinach para | Cucumber and yogurt sa
 ✅ dinner      628/ 600  [wheat   ]  Arhar with spinach (Ar | Stuffed okra (Bharwa b | Chapati/Roti | Curd mint dip
 ✅ snack       182/ 200  [-       ]  Cabbage rolls (curry) 

 DAY 3  |  GLUCOSE: 203 mg/dL
 KCAL: 1941/2000 (97.0%)
 ✅ breakfast   356/ 500  [millet  ]  Jowar dosa | Rolled sandwich
 ✅ lunch       822/ 700  [wheat   ]

In [17]:
# ╔══════════════════════════════════════╗
N_DAYS     = 7
DAILY_KCAL = 2000
DIET       = 'egg'   # 🔁 change to 'nonveg' / 'mixed' later
# ╚══════════════════════════════════════╝

import random
from collections import Counter

# realistic glucose fluctuation (can tweak)
def generate_glucose_series(n):
    base = 220
    return [max(100, int(base + random.randint(-40, 60))) for _ in range(n)]

glucose_series = generate_glucose_series(N_DAYS)

week_used  = {}
all_foods  = []
daily_pcts = []
grain_log  = []

print(f"\n🧪 TEST RUN: {DIET.upper()} DIET | {N_DAYS} DAYS\n")

for day in range(N_DAYS):
    
    glucose = glucose_series[day]

    plan = generate_daily_plan(
        glucose_mg_dl = glucose,
        daily_kcal    = DAILY_KCAL,
        diet          = DIET,
        week_used     = week_used,
        seed          = None
    )

    week_used = plan['week_used']
    s         = plan['summary']
    daily_pcts.append(s['pct_achieved'])

    print(f"\n{'='*70}")
    print(f" DAY {day+1}  |  GLUCOSE: {glucose} mg/dL")
    print(f" KCAL: {s['total_kcal']:.0f}/{s['daily_kcal_target']:.0f} ({s['pct_achieved']:.1f}%)")
    print(f"{'='*70}")

    for slot in ['breakfast','lunch','dinner','snack']:
        meal  = plan[slot]
        got   = s['meal_kcal'][slot]
        tgt   = s['kcal_targets'][slot]
        valid = validate_meal(meal, slot)
        badge = '✅' if valid else '⚠️'

        # grain tracking
        carb_row = meal[meal['food_role'].isin(
            ['roti','rice','breakfast_carb'])] if not meal.empty else None
        
        grain = carb_row.iloc[0]['grain_category'] if carb_row is not None and not carb_row.empty else '-'
        
        if slot in ['breakfast','lunch','dinner']:
            grain_log.append(grain)

        foods = ' | '.join(
            meal['food_name'].str[:22].tolist()) if not meal.empty else 'EMPTY'

        print(f" {badge} {slot:10s} {got:4.0f}/{tgt:4.0f}  [{grain:8s}]  {foods}")

        if not meal.empty:
            all_foods.extend(meal['food_name'].tolist())

# ─────────────── REPORT ───────────────

counts   = Counter(all_foods)
repeated = {k: v for k, v in counts.items() if v > 1}

print(f"\n{'─'*70}")
print(f" 📊 WEEKLY REPORT ({DIET})")
print(f" Total meals        : {len(all_foods)}")
print(f" Unique foods       : {len(counts)}")
print(f" Repeated foods     : {len(repeated)}")
print(f" Avg kcal achieved  : {sum(daily_pcts)/len(daily_pcts):.1f}%")

if repeated:
    print("\n 🔁 Top Repeated Foods:")
    for k, v in sorted(repeated.items(), key=lambda x: -x[1])[:8]:
        print(f"   {k[:50]:50s} : {v}x")
else:
    print("\n ✅ No repetition detected")

print(f"\n 🌾 Grain Rotation:")
print("   " + " → ".join(grain_log[:12]) + " ...")


🧪 TEST RUN: EGG DIET | 7 DAYS


 DAY 1  |  GLUCOSE: 197 mg/dL
 KCAL: 1846/2000 (92.3%)
 ✅ breakfast   462/ 500  [rice    ]  Idli | Jackfruit fritters (Po | Fried Egg
 ✅ lunch       621/ 700  [wheat   ]  Boiled egg (Ubla anda) | Mushroom matar | Stuffed okra (Bharwa b | Besan and spinach para | Curd vegetable dip
 ✅ dinner      580/ 600  [wheat   ]  Scrambled egg (Ande ki | Soyabean curry | Bottle gourd raita (Gh | Tandoori parantha/para | Curd mint dip
 ✅ snack       183/ 200  [-       ]  Cabbage rolls (dry) (( | Rolled sandwich

 DAY 2  |  GLUCOSE: 274 mg/dL
 KCAL: 1805/2000 (90.2%)
 ✅ breakfast   415/ 500  [rice    ]  Instant idli (with sem | Dhokla | Poached egg | Curd vegetable dip
 ✅ lunch       619/ 700  [wheat   ]  Indian style egg bhuji | Black channa curry/Ben | Carrot and fenugreek l | Chapati/Roti
 ✅ dinner      576/ 600  [wheat   ]  Fried Egg | Whole masoor (Masoor k | Bottle gourd soup (Ghi | Paushtik roti
 ✅ snack       196/ 200  [-       ]  Khaman (dhokla) | Soya seekh 

In [18]:
# ╔══════════════════════════════════════╗
N_DAYS     = 7
DAILY_KCAL = 2200
DIET       = 'nonveg'   # 🔁 change to 'nonveg' / 'mixed' later
# ╚══════════════════════════════════════╝

import random
from collections import Counter

# realistic glucose fluctuation (can tweak)
def generate_glucose_series(n):
    base = 190
    return [max(100, int(base + random.randint(-40, 60))) for _ in range(n)]

glucose_series = generate_glucose_series(N_DAYS)

week_used  = {}
all_foods  = []
daily_pcts = []
grain_log  = []

print(f"\n🧪 TEST RUN: {DIET.upper()} DIET | {N_DAYS} DAYS\n")

for day in range(N_DAYS):
    
    glucose = glucose_series[day]

    plan = generate_daily_plan(
        glucose_mg_dl = glucose,
        daily_kcal    = DAILY_KCAL,
        diet          = DIET,
        week_used     = week_used,
        seed          = None
    )

    week_used = plan['week_used']
    s         = plan['summary']
    daily_pcts.append(s['pct_achieved'])

    print(f"\n{'='*70}")
    print(f" DAY {day+1}  |  GLUCOSE: {glucose} mg/dL")
    print(f" KCAL: {s['total_kcal']:.0f}/{s['daily_kcal_target']:.0f} ({s['pct_achieved']:.1f}%)")
    print(f"{'='*70}")

    for slot in ['breakfast','lunch','dinner','snack']:
        meal  = plan[slot]
        got   = s['meal_kcal'][slot]
        tgt   = s['kcal_targets'][slot]
        valid = validate_meal(meal, slot)
        badge = '✅' if valid else '⚠️'

        # grain tracking
        carb_row = meal[meal['food_role'].isin(
            ['roti','rice','breakfast_carb'])] if not meal.empty else None
        
        grain = carb_row.iloc[0]['grain_category'] if carb_row is not None and not carb_row.empty else '-'
        
        if slot in ['breakfast','lunch','dinner']:
            grain_log.append(grain)

        foods = ' | '.join(
            meal['food_name'].str[:22].tolist()) if not meal.empty else 'EMPTY'

        print(f" {badge} {slot:10s} {got:4.0f}/{tgt:4.0f}  [{grain:8s}]  {foods}")

        if not meal.empty:
            all_foods.extend(meal['food_name'].tolist())

# ─────────────── REPORT ───────────────

counts   = Counter(all_foods)
repeated = {k: v for k, v in counts.items() if v > 1}

print(f"\n{'─'*70}")
print(f" 📊 WEEKLY REPORT ({DIET})")
print(f" Total meals        : {len(all_foods)}")
print(f" Unique foods       : {len(counts)}")
print(f" Repeated foods     : {len(repeated)}")
print(f" Avg kcal achieved  : {sum(daily_pcts)/len(daily_pcts):.1f}%")

if repeated:
    print("\n 🔁 Top Repeated Foods:")
    for k, v in sorted(repeated.items(), key=lambda x: -x[1])[:8]:
        print(f"   {k[:50]:50s} : {v}x")
else:
    print("\n ✅ No repetition detected")

print(f"\n 🌾 Grain Rotation:")
print("   " + " → ".join(grain_log[:12]) + " ...")


🧪 TEST RUN: NONVEG DIET | 7 DAYS


 DAY 1  |  GLUCOSE: 225 mg/dL
 KCAL: 2025/2200 (92.1%)
 ✅ breakfast   363/ 550  [rice    ]  Semolina idli (Suji/Ra | Cabbage rolls (dry) ((
 ✅ lunch       875/ 770  [wheat   ]  Tandoori fish | Spinach paneer (Palak  | Sarson ka saag | Chapati/Roti
 ✅ dinner      588/ 660  [millet  ]  Keema kofta curry | Carrot and fenugreek l | Makki ki roti
 ✅ snack       199/ 220  [-       ]  Indian style egg bhuji | Poached egg

 DAY 2  |  GLUCOSE: 204 mg/dL
 KCAL: 2160/2200 (98.2%)
 ✅ breakfast   438/ 550  [semolina]  Lentils and wheat porr | Pin wheel sandwich | Fish tikka
 ✅ lunch       865/ 770  [wheat   ]  Tomato chicken | Black channa curry/Ben | Bottle gourd soup (Ghi | Plain parantha/paratha
 ✅ dinner      660/ 660  [wheat   ]  Fried chicken with tom | Bottle gourd raita (Gh | Besan and spinach para
 ✅ snack       198/ 220  [-       ]  Fried Egg | Boiled egg (Ubla anda)

 DAY 3  |  GLUCOSE: 154 mg/dL
 KCAL: 2403/2200 (109.2%)
 ✅ breakfast   663/ 550  [rice

In [19]:
# ╔══════════════════════════════════════╗
N_DAYS     = 7
DAILY_KCAL = 2600
DIET       = 'veg'   # 🔁 change to 'nonveg' / 'mixed' later
# ╚══════════════════════════════════════╝

import random
from collections import Counter

# realistic glucose fluctuation (can tweak)
def generate_glucose_series(n):
    base = 160
    return [max(100, int(base + random.randint(-40, 60))) for _ in range(n)]

glucose_series = generate_glucose_series(N_DAYS)

week_used  = {}
all_foods  = []
daily_pcts = []
grain_log  = []

print(f"\n🧪 TEST RUN: {DIET.upper()} DIET | {N_DAYS} DAYS\n")

for day in range(N_DAYS):
    
    glucose = glucose_series[day]

    plan = generate_daily_plan(
        glucose_mg_dl = glucose,
        daily_kcal    = DAILY_KCAL,
        diet          = DIET,
        week_used     = week_used,
        seed          = None
    )

    week_used = plan['week_used']
    s         = plan['summary']
    daily_pcts.append(s['pct_achieved'])

    print(f"\n{'='*70}")
    print(f" DAY {day+1}  |  GLUCOSE: {glucose} mg/dL")
    print(f" KCAL: {s['total_kcal']:.0f}/{s['daily_kcal_target']:.0f} ({s['pct_achieved']:.1f}%)")
    print(f"{'='*70}")

    for slot in ['breakfast','lunch','dinner','snack']:
        meal  = plan[slot]
        got   = s['meal_kcal'][slot]
        tgt   = s['kcal_targets'][slot]
        valid = validate_meal(meal, slot)
        badge = '✅' if valid else '⚠️'

        # grain tracking
        carb_row = meal[meal['food_role'].isin(
            ['roti','rice','breakfast_carb'])] if not meal.empty else None
        
        grain = carb_row.iloc[0]['grain_category'] if carb_row is not None and not carb_row.empty else '-'
        
        if slot in ['breakfast','lunch','dinner']:
            grain_log.append(grain)

        foods = ' | '.join(
            meal['food_name'].str[:22].tolist()) if not meal.empty else 'EMPTY'

        print(f" {badge} {slot:10s} {got:4.0f}/{tgt:4.0f}  [{grain:8s}]  {foods}")

        if not meal.empty:
            all_foods.extend(meal['food_name'].tolist())

# ─────────────── REPORT ───────────────

counts   = Counter(all_foods)
repeated = {k: v for k, v in counts.items() if v > 1}

print(f"\n{'─'*70}")
print(f" 📊 WEEKLY REPORT ({DIET})")
print(f" Total meals        : {len(all_foods)}")
print(f" Unique foods       : {len(counts)}")
print(f" Repeated foods     : {len(repeated)}")
print(f" Avg kcal achieved  : {sum(daily_pcts)/len(daily_pcts):.1f}%")

if repeated:
    print("\n 🔁 Top Repeated Foods:")
    for k, v in sorted(repeated.items(), key=lambda x: -x[1])[:8]:
        print(f"   {k[:50]:50s} : {v}x")
else:
    print("\n ✅ No repetition detected")

print(f"\n 🌾 Grain Rotation:")
print("   " + " → ".join(grain_log[:12]) + " ...")


🧪 TEST RUN: VEG DIET | 7 DAYS


 DAY 1  |  GLUCOSE: 184 mg/dL
 KCAL: 2064/2600 (79.4%)
 ✅ breakfast   331/ 650  [rice    ]  Semolina idli (Suji/Ra | Pin wheel sandwich
 ✅ lunch       752/ 910  [wheat   ]  Mushroom matar | Carrot and fenugreek l | Paushtik roti | Curd vegetable dip
 ✅ dinner      731/ 780  [wheat   ]  Soyabean curry | Bottle gourd soup (Ghi | Besan and spinach para | Curd mint dip
 ✅ snack       250/ 260  [-       ]  Soya seekh kebab | Cabbage rolls (dry) ((

 DAY 2  |  GLUCOSE: 197 mg/dL
 KCAL: 2119/2600 (81.5%)
 ✅ breakfast   467/ 650  [millet  ]  Jowar dosa | Jackfruit fritters (Po | Curd vegetable dip
 ✅ lunch       717/ 910  [wheat   ]  Black channa curry/Ben | Bottle gourd raita (Gh | Methi thepla | Mint raita (Pudinay ka
 ✅ dinner      691/ 780  [wheat   ]  Whole masoor (Masoor k | Sarson ka saag | Chapati/Roti
 ✅ snack       244/ 260  [-       ]  Semolina upma (Suji/Ra | Dhokla

 DAY 3  |  GLUCOSE: 123 mg/dL
 KCAL: 2245/2600 (86.3%)
 ✅ breakfast   482/ 650  [ri

In [20]:
# ╔══════════════════════════════════════╗
N_DAYS     = 7
DAILY_KCAL = 2500
DIET       = 'nonveg'   # 🔁 change to 'nonveg' / 'mixed' later
# ╚══════════════════════════════════════╝

import random
from collections import Counter

# realistic glucose fluctuation (can tweak)
def generate_glucose_series(n):
    base = 150
    return [max(100, int(base + random.randint(-20, 30))) for _ in range(n)]

glucose_series = generate_glucose_series(N_DAYS)

week_used  = {}
all_foods  = []
daily_pcts = []
grain_log  = []

print(f"\n🧪 TEST RUN: {DIET.upper()} DIET | {N_DAYS} DAYS\n")

for day in range(N_DAYS):
    
    glucose = glucose_series[day]

    plan = generate_daily_plan(
        glucose_mg_dl = glucose,
        daily_kcal    = DAILY_KCAL,
        diet          = DIET,
        week_used     = week_used,
        seed          = None
    )

    week_used = plan['week_used']
    s         = plan['summary']
    daily_pcts.append(s['pct_achieved'])

    print(f"\n{'='*70}")
    print(f" DAY {day+1}  |  GLUCOSE: {glucose} mg/dL")
    print(f" KCAL: {s['total_kcal']:.0f}/{s['daily_kcal_target']:.0f} ({s['pct_achieved']:.1f}%)")
    print(f"{'='*70}")

    for slot in ['breakfast','lunch','dinner','snack']:
        meal  = plan[slot]
        got   = s['meal_kcal'][slot]
        tgt   = s['kcal_targets'][slot]
        valid = validate_meal(meal, slot)
        badge = '✅' if valid else '⚠️'

        # grain tracking
        carb_row = meal[meal['food_role'].isin(
            ['roti','rice','breakfast_carb'])] if not meal.empty else None
        
        grain = carb_row.iloc[0]['grain_category'] if carb_row is not None and not carb_row.empty else '-'
        
        if slot in ['breakfast','lunch','dinner']:
            grain_log.append(grain)

        foods = ' | '.join(
            meal['food_name'].str[:22].tolist()) if not meal.empty else 'EMPTY'

        print(f" {badge} {slot:10s} {got:4.0f}/{tgt:4.0f}  [{grain:8s}]  {foods}")

        if not meal.empty:
            all_foods.extend(meal['food_name'].tolist())

# ─────────────── REPORT ───────────────

counts   = Counter(all_foods)
repeated = {k: v for k, v in counts.items() if v > 1}

print(f"\n{'─'*70}")
print(f" 📊 WEEKLY REPORT ({DIET})")
print(f" Total meals        : {len(all_foods)}")
print(f" Unique foods       : {len(counts)}")
print(f" Repeated foods     : {len(repeated)}")
print(f" Avg kcal achieved  : {sum(daily_pcts)/len(daily_pcts):.1f}%")

if repeated:
    print("\n 🔁 Top Repeated Foods:")
    for k, v in sorted(repeated.items(), key=lambda x: -x[1])[:8]:
        print(f"   {k[:50]:50s} : {v}x")
else:
    print("\n ✅ No repetition detected")

print(f"\n 🌾 Grain Rotation:")
print("   " + " → ".join(grain_log[:12]) + " ...")


🧪 TEST RUN: NONVEG DIET | 7 DAYS


 DAY 1  |  GLUCOSE: 174 mg/dL
 KCAL: 2465/2500 (98.6%)
 ✅ breakfast   572/ 625  [rice    ]  Moong bean dosa (Pesar | Hariyali Fish Tikka
 ✅ lunch       868/ 875  [wheat   ]  Tomato chicken | Black channa curry/Ben | Sarson ka saag | Cauliflower parantha/p
 ✅ dinner      777/ 750  [millet  ]  Tandoori fish | Stuffed okra (Bharwa b | Makki ki roti
 ✅ snack       248/ 250  [-       ]  Poached egg | Scrambled egg (Ande ki

 DAY 2  |  GLUCOSE: 164 mg/dL
 KCAL: 2023/2500 (80.9%)
 ✅ breakfast   576/ 625  [rice    ]  Idli | Boti kebab | Fried Egg
 ✅ lunch       807/ 875  [wheat   ]  Roast chicken | Soyabean curry | Cabbage and peas (Patt | Keema parantha/paratha
 ✅ dinner      428/ 750  [wheat   ]  Keema kofta curry | Bottle gourd raita (Gh | Paushtik roti
 ✅ snack       213/ 250  [-       ]  Fish tikka | Indian style egg bhuji

 DAY 3  |  GLUCOSE: 156 mg/dL
 KCAL: 2429/2500 (97.2%)
 ✅ breakfast   635/ 625  [semolina]  Lentils and wheat porr | Cabbage rolls 